# İstanbul Raylı Sistem Yolcu Ağı — Analiz Defteri

Bu defter proje betiklerinin ürettiği gerçek çıktıları kontrol etmek ve temel sonuçları yorumlamak için hazırlanmıştır. Çıktılar güncel değilse proje kökünde `python run_project.py` komutunu çalıştır.

**Önemli sınırlar:** Kaynak veri bir OD (başlangıç-varış) matrisi değildir. Kenar ağırlığı yolcu aktivitesi vekilidir; kapanma senaryosu gerçek yolcu yönlenmesini kanıtlamaz. Genel yönergedeki 500 düğüm şartı ayrıca öğretim elemanıyla netleştirilmelidir.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import networkx as nx

ROOT = Path.cwd()
if not (ROOT / 'outputs' / 'analysis_summary.json').exists():
    ROOT = ROOT.parent  # Notebook notebooks/ içinden açıldıysa proje kökü
SUMMARY_PATH = ROOT / 'outputs' / 'analysis_summary.json'
AUDIT_PATH = ROOT / 'outputs' / 'data_audit.json'
GRAPH_PATH = ROOT / 'outputs' / 'istanbul_rail_2025.graphml'
assert SUMMARY_PATH.exists(), 'Önce proje kökünde python run_project.py komutunu çalıştır.'
summary = json.loads(SUMMARY_PATH.read_text(encoding='utf-8'))
audit = json.loads(AUDIT_PATH.read_text(encoding='utf-8'))
G = nx.Graph(nx.read_graphml(GRAPH_PATH))
print('Kaynak satırı:', audit['rows'])
print('Ham hat sayısı:', audit['distinct_lines'])
print('Düğüm / kenar:', G.number_of_nodes(), '/', G.number_of_edges())
print('Bileşen sayısı:', nx.number_connected_components(G))

## 1. Veri kalite özeti
Aşağıdaki tablo kaynağın eksik alanlarını özetler. Eksik istasyon adları ve koordinatlar otomatik olarak uydurulmaz.


In [ ]:
pd.DataFrame([
    ('Kaynak kayıt sayısı', audit['rows']),
    ('Kaynak sütun sayısı', audit['columns']),
    ('İstasyon adı boş kayıt', audit['missing_station_name_records']),
    ('Boylamı boş kayıt', audit['missing_longitude_source_records']),
    ('Enlemi boş kayıt', audit['missing_latitude_source_records']),
    ('Koordinatı aralık kontrolünden geçen kayıt', audit['normalized_valid_coordinate_records']),
    ('Tam yinelenen satır', audit['duplicate_full_source_rows']),
], columns=['Kontrol', 'Değer'])

## 2. Ağın yapısı
Ortalama yol uzunluğu ve çap yalnızca en büyük bağlı bileşen üzerinde yorumlanır; tüm grafik bağlı değildir.


In [ ]:
g = summary['graph']
pd.DataFrame([
    ('Düğüm', g['nodes']), ('Kenar', g['edges']), ('Yoğunluk', g['density']),
    ('Bağlı bileşen', g['connected_components']), ('Bileşen boyutları', ', '.join(map(str, g['component_sizes']))),
    ('En büyük bileşen düğüm sayısı', g['largest_component_nodes']),
    ('En büyük bileşen ortalama yolu', g['average_shortest_path_largest_component']),
    ('En büyük bileşen çapı', g['diameter_largest_component']),
    ('Ortalama kümelenme', g['average_clustering_all_graph']),
    ('Derece benzeşimi', g['degree_assortativity']),
], columns=['Ölçü', 'Değer'])

## 3. Merkeziyet karşılaştırması
Ağırlıklı merkeziyet için `distance = 1 / strength` maliyeti kullanılır. `strength`, doğrudan OD yolculuk akışı değil, iki uç istasyonun ortalama günlük yolcu aktivitesinden türetilmiş vekildir.


In [ ]:
metrics_path = ROOT / 'outputs' / 'centrality_and_communities.csv'
m = pd.read_csv(metrics_path)
cols = ['label', 'lines', 'degree', 'betweenness_unweighted', 'betweenness_weighted', 'passenger_mean_daily']
display(m.sort_values('betweenness_unweighted', ascending=False)[cols].head(10))
print('Spearman çift listesi: outputs/centrality_spearman_correlations.csv')
pd.read_csv(ROOT / 'outputs' / 'centrality_spearman_correlations.csv').head(10)

## 4. Topluluklar
Aşağıdaki adlar, toplulukların hat üyeliklerinden türetilen çalışma adlarıdır; resmî veya yerleşim tabanlı sınıflar olarak kabul edilmemelidir.


In [ ]:
profile_path = ROOT / 'outputs' / 'community_profile.csv'
if profile_path.exists():
    profile = pd.read_csv(profile_path)
    display(profile[['rank_by_size', 'community', 'candidate_name', 'size', 'dominant_lines']].head(10))
else:
    print('community_profile.csv bulunamadı; proje analizini yeniden çalıştır.')

## 5. Null model ve kişisel yöntemlerin durumu
100 ağla karşılaştırma, gözlenen ölçünün derece-koruyan rastgele ağ dağılımından ne kadar ayrıldığını değerlendirir. Null modeli amaçlanan anlamıyla yorumla; z-skoru tek başına nedensellik göstermez.


In [ ]:
null = summary['null_model']
print('Null model durumu:', null.get('status'))
print('Tekrar:', null.get('replicates_completed'), '/', null.get('replicates_requested'))
if null.get('status') == 'calculated':
    display(pd.DataFrame(null['metrics']).T)
print('Leiden durumu:', summary['community_detection']['leiden'].get('status'))
print('Power-law durumu:', summary['powerlaw'].get('status'))
if summary['powerlaw'].get('status') == 'calculated':
    print('alpha:', summary['powerlaw'].get('alpha'), 'xmin:', summary['powerlaw'].get('xmin'))
    print('Power-law vs exponential:', summary['powerlaw'].get('power_law_vs_exponential'))
    print('Power-law vs lognormal:', summary['powerlaw'].get('power_law_vs_lognormal'))

## 6. İstasyon kapanması — yorum sınırı
Bu senaryo Sirkeci düğümünü çıkarır ve bileşen/yol/merkeziyet değişimini karşılaştırır. Kaynakta gerçek yolcu başlangıç-varış eşleşmeleri bulunmadığı için yolcu yükünün hangi hatta ne kadar kayacağı hesaplanamaz. Sonuç topolojik hassasiyet senaryosudur.

## Teslim öncesi mutlaka kapat
- `docs/topology_validation.csv` içinde kalan hat/aktarma kontrollerini yap.
- `data/processed/station_alias_review.csv` ve `outputs/unmatched_source_stations.csv` dosyalarını gözden geçir.
- Kaynak portalın lisans/atıf koşulunu doğrula; `docs/data_dictionary.csv` içindeki metaveri notunu güncelle.
- Öğretim elemanından 500 düğüm şartının bu fiziksel istasyon ağına nasıl uygulanacağını yazılı olarak netleştir. Düğüm uydurma.
- `analysis_summary.json` içinde Leiden ve power-law durumunun `calculated` olduğuna bak; değilse tamamlandı deme.
- `web/interactive_network.html` içinde arama, düğüm detayı ve topluluk filtresini test et.
